# Projeto de Integração - Pipeline ELT
Este notebook realiza a Extração, Carga (EL) e Transformação (T) dos dados de Transferências de Recursos Federais.
Para contornar limitações de memória (RAM) e espaço, o pipeline configura um banco de dados PostgreSQL local na máquina virtual do Colab e utiliza o **dbt** para a modelagem dimensional (Esquema Estrela).

In [ ]:
import os
from google.colab import drive

print("📁 1. Conectando ao Google Drive para ler os Dados Brutos...")
drive.mount('/content/drive')

print("\n📥 2. Clonando apenas o CÓDIGO do repositório (Bypass no limite do LFS)...")
# O comando GIT_LFS_SKIP_SMUDGE=1 faz o Git baixar a pasta transferencias_elt perfeitamente, ignorando os CSVs pesados
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/jl-hm/projeto_integracao.git
%cd projeto_integracao

print("\n🐘 3. Instalando o PostgreSQL local e o dbt...")
!sudo apt-get -y -qq update
!sudo apt-get -y -qq install postgresql
!sudo service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'suasenhaaqui';"
!pip install --no-cache-dir dbt-core==1.8.2 dbt-postgres==1.8.2 -q

os.environ['PATH'] = "/usr/local/bin:/root/.local/bin:" + os.environ['PATH']
print("\n✅ Ambiente configurado com sucesso!")

📁 1. Conectando ao Google Drive para ler os Dados Brutos...
Mounted at /content/drive

📥 2. Clonando apenas o CÓDIGO do repositório (Bypass no limite do LFS)...
fatal: destination path 'projeto_integracao' already exists and is not an empty directory.
/content/projeto_integracao/transferencias_elt/projeto_integracao

🐘 3. Instalando o PostgreSQL local e o dbt...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting PostgreSQL 14 database server
   ...done.
ALTER ROLE

✅ Ambiente configurado com sucesso!


In [ ]:
import pandas as pd
from sqlalchemy import create_engine


usuario_db = 'postgres'
senha_db = 'suasenhaaqui'
host_db = 'localhost'
porta_db = '5432'
nome_banco_db = 'postgres'

DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}'
engine = create_engine(DATABASE_URL)

print("🔌 Conexão com o PostgreSQL Local estabelecida com sucesso!")

🔌 Conexão com o PostgreSQL Local estabelecida com sucesso!


In [ ]:
import pandas as pd
import os

print("⏳ Iniciando processo de Extração e Carga Bruta (EL) em Lotes...")

# --- Conexão com o Google Drive  ---
base_path = '/content/drive/MyDrive/trab_rt/Dados_Brutos'
anos = ['2020', '2021', '2022']
meses = [str(m).zfill(2) for m in range(1, 13)]

nome_tabela = 'raw_transferencias'
primeira_insercao = True

for ano in anos:
    pasta_ano = f"{ano}_rt"

    for mes in meses:
        nome_arquivo = f"{ano}{mes}_Transferencias.csv"
        caminho_arquivo = f"{base_path}/{pasta_ano}/{nome_arquivo}"

        try:
            df_raw = pd.read_csv(caminho_arquivo, sep=';', encoding='latin1', on_bad_lines='skip', dtype=str)
            acao_banco = 'replace' if primeira_insercao else 'append'

            df_raw.to_sql(nome_tabela, con=engine, if_exists=acao_banco, index=False, chunksize=10000)
            print(f"🚀 Sucesso! Mês {mes}/{ano} empilhado na tabela '{nome_tabela}'.")
            primeira_insercao = False

        except FileNotFoundError:
            print(f"⚠️ Aviso: Arquivo não encontrado no Drive: {caminho_arquivo}")
        except Exception as e:
            print(f"❌ Erro ao processar o arquivo: {e}")
            engine.dispose()

print("\n🎯 Fase EL Concluída! Registros carregados na área de Staging.")

⏳ Iniciando processo de Extração e Carga Bruta (EL) em Lotes...
🚀 Sucesso! Mês 01/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 02/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 03/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 04/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 05/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 06/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 07/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 08/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 09/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 10/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 11/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 12/2020 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 01/2021 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 02/2021 empilhado na tabela 'raw_transferencias'.
🚀 Sucesso! Mês 03/2021 emp

In [ ]:
print("🔨 Iniciando a Fase de Transformação (T) com o dbt...")

# Navega para a pasta do dbt dentro do repositório
dbt_path = "transferencias_elt"
%cd {dbt_path}


print("⚙️ Gerando o arquivo de senhas (profiles.yml) dinamicamente...")
profiles_content = """
transferencias_elt:
  target: dev
  outputs:
    dev:
      type: postgres
      host: localhost
      user: postgres
      password: suasenhaaqui
      port: 5432
      dbname: postgres
      schema: public
      threads: 4
"""

# Salva o arquivo na pasta padrão que o dbt procura
os.makedirs('/root/.dbt', exist_ok=True)
with open('/root/.dbt/profiles.yml', 'w') as f:
    f.write(profiles_content.strip())



# Garante que o banco está rodando antes do dbt executar
!sudo service postgresql start

print("\n🔍 Testando a configuração e conexão (dbt debug)...")
!dbt debug

print("\n🌟 Construindo o Data Warehouse (dbt run)...")
!dbt run

print("\n🏆 Pipeline ELT concluído! Esquema Estrela gerado com sucesso no banco de dados.")

🔨 Iniciando a Fase de Transformação (T) com o dbt...
/content/projeto_integracao/transferencias_elt/projeto_integracao/transferencias_elt
⚙️ Gerando o arquivo de senhas (profiles.yml) dinamicamente...
 * Starting PostgreSQL 14 database server
   ...done.

🔍 Testando a configuração e conexão (dbt debug)...
17:27:15  Running with dbt=1.8.2
17:27:15  dbt version: 1.8.2
17:27:15  python version: 3.12.13
17:27:15  python path: /usr/bin/python3
17:27:15  os info: Linux-6.6.122+-x86_64-with-glibc2.35
17:27:16  Using profiles dir at /root/.dbt
17:27:16  Using profiles.yml file at /root/.dbt/profiles.yml
17:27:16  Using dbt_project.yml file at /content/projeto_integracao/transferencias_elt/projeto_integracao/transferencias_elt/dbt_project.yml
17:27:16  adapter type: postgres
17:27:16  adapter version: 1.8.2
17:27:16  Configuration:
17:27:16    profiles.yml file [OK found and valid]
17:27:16    dbt_project.yml file [OK found and valid]
17:27:16  Required dependencies:
17:27:16   - git [OK found]